# Two-group PPI: striatum–auditory cortex coupling by learning stage

ROI-to-ROI generalized PPI (`ppi_roi.py`): does coupling between the anterior caudate and auditory cortex depend on the task event, does it differ between learners (non-Mandarin, n = 12) and natives (Mandarin, n = 13), and does it change with learning?

**Model** (per participant × stage × seed; details in the `ppi_roi.py` docstring): the univariate `scrubbed-splitfb` design (sound, resp, fb_correct, fb_wrong, fb_noresp, drift, motion + aCompCor, FD/DVARS scrubbing), plus the seed time series (cleaned of drift and confounds, not task) and seed × condition interaction terms. There's no deconvolution (TR = 3 s). resp has no interaction term: it follows the sound by <1 s, so its term would nearly duplicate the sound term. Target time series are z-scored within run; runs are combined within stage by fixed effects.

**Seeds:** bilateral aCAU (main) and bilateral aPUT (comparison: the univariate feedback result is aCAU vs aPUT). **Targets:** the 14 auditory ROIs, and the three composites from the multivariate analyses (`multivariate_data.COMPOSITES`: Primary = HG, PT, PP; Association = STGa, STGp; IFG = pars opercularis, pars triangularis).

**Contrasts:**
- `seed`: task-independent coupling (sanity check)
- `ppifbcorrect`: coupling during correct feedback, both groups (**planned**)
- `ppisound`: coupling during the sound (+ response) period, both groups
- `ppifbwrong`: coupling during wrong feedback, learners only (natives have ~1 wrong trial per run)
- `ppifbcorrectvswrong`: learners only

**Planned tests:** aCAU seed, `ppifbcorrect`, three composites: learners − natives with stages averaged, and the learning-stage trend [−1, 0, 1] learners − natives; FDR across the three composites (`group_stats`, as in the univariate and multivariate notebooks). Everything else is secondary or exploratory.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.join('..', '10_multivariate'))
import group_stats as gs
import multivariate_data as mvd

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 24)

nilearn_dir = '/ix1/bchandrasekaran/krs228/data/FLT/data_denoised/derivatives/nilearn'
group_dir = os.path.join(nilearn_dir, 'group_space-T1w_fwhm-0.00')
out_dir = os.path.join(group_dir, 'stats_two-group', 'ppi')
os.makedirs(out_dir, exist_ok=True)

seeds = ('aCAU', 'aPUT')
contrasts = ('seed', 'ppisound', 'ppifbcorrect', 'ppifbwrong', 'ppifbcorrectvswrong')
composite_network = 'auditory-composite'
composites = list(mvd.COMPOSITES)

# resampling settings, matching robustness_checks.py
n_perms, n_boot, seed = 10000, 10000, 0
colors = {'non-Mandarin': '#2a6fdb', 'Mandarin': '#d9822b'}   # as in group_level_two_group.ipynb
labels = {'non-Mandarin': 'learners', 'Mandarin': 'natives'}


def per_participant(df):
    # one value per participant x contrast x region: hemispheres, then stages, averaged
    return (gs.bilateral_mean(df).groupby(['contrast', 'region', 'participant_id', 'participant_group'],
                                          as_index=False).beta.mean())


def robust_outliers(values):
    # participants more than 3 robust SDs (1.4826 x MAD) from their group median
    def flag(v):
        med = v.median()
        mad = 1.4826 * (v - med).abs().median()
        return (v - med).abs() > 3 * mad if mad > 0 else v * False
    return values[values.groupby(['contrast', 'region', 'participant_group']).beta.transform(flag).astype(bool)]


def participant_strip(ax, values, title):
    # per-participant values by group, group mean +/- SEM, robust outliers labeled
    outliers = set(robust_outliers(values).participant_id)
    for x, group in enumerate(gs.GROUPS):
        v = values[values.participant_group == group]
        jitter = np.random.default_rng(x).uniform(-0.12, 0.12, len(v))
        ax.scatter(x + jitter, v.beta, s=16, color=colors[group], alpha=0.75)
        ax.errorbar(x + 0.3, v.beta.mean(), yerr=v.beta.sem(), fmt='o', color='k', ms=4, capsize=3)
        for j, (_, row) in zip(jitter, v.iterrows()):
            if row.participant_id in outliers:
                ax.annotate(row.participant_id.replace('sub-', ''), (x + j, row.beta), fontsize=6,
                            xytext=(3, 0), textcoords='offset points')
    ax.axhline(0, color='0.6', lw=0.8)
    ax.set_xticks([0, 1], [labels[g] for g in gs.GROUPS])
    ax.set_title(title, fontsize=8)

## 1. Load and check

In [ ]:
ppi = {s: mvd.add_composites(gs.load_roi_values(os.path.join(group_dir, f'roi-values_ppi-seed-{s}'),
                                                networks=('auditory',), contrasts=contrasts))
       for s in seeds}
qc = pd.concat([pd.read_csv(os.path.join(group_dir, f'roi-values_ppi-seed-{s}', 'ppi-qc.tsv'), sep='\t')
                for s in seeds], ignore_index=True)

for s in seeds:
    print(f'--- {s} seed: participants per contrast x group x stage')
    display(gs.coverage(ppi[s]))

In [ ]:
# per run: retained volumes, residual dof, and how collinear the interaction terms are
# (feedback is one TR after the sound, so ppi_sound and ppi_fb_* share variance)
vif_cols = [c for c in qc.columns if c.startswith('vif_')]
display(qc[qc.seed == 'aCAU'].groupby('participant_group')[['n_retained', 'residual_dof', 'seed_n_voxels']
                                                            + vif_cols].describe().T.round(1))
# natives' wrong-feedback trials per stage (why ppifbcorrectvswrong is learners only)
display(qc[qc.seed == 'aCAU'].groupby(['participant_group', 'learning_stage'])['n_fb_wrong'].describe()
        [['mean', 'min', 'max']].round(1))

# runs left out for too few residual dof (ppi_roi.py MIN_RESIDUAL_DOF); a stage with none left is missing
dropped = qc[(qc.seed == 'aCAU') & ~qc.included.astype(bool)]
print(f'{len(dropped)} runs left out')
dropped[['participant_id', 'participant_group', 'learning_stage', 'run', 'n_volumes', 'n_retained', 'residual_dof']]

## 2. Task-independent coupling (sanity check)

The seed term should be positive across auditory cortex in both groups: the ROIs share low-frequency fluctuation even after confounds. If it isn't, something upstream is wrong (masks, seed extraction), and the PPI terms can't be trusted.

In [ ]:
seed_coupling = gs.baseline_tests(ppi['aCAU'][ppi['aCAU'].contrast == 'seed'])
seed_coupling[['network', 'participant_group', 'region', 'n', 'mean', 't', 'p', 'q']].round(3)

### Per-participant check

Every participant's value (hemispheres and stages averaged), aCAU seed, composites. Participants more than 3 robust SDs from their group's median are labeled and listed. One extreme participant can carry a group mean at n = 12, so look here before reading any test.

In [ ]:
check_contrasts = ['seed', 'ppisound', 'ppifbcorrect']
values = per_participant(ppi['aCAU'][ppi['aCAU'].network == composite_network])
fig, axes = plt.subplots(len(check_contrasts), len(composites), figsize=(3 * len(composites), 2.6 * len(check_contrasts)))
for row, contrast in zip(axes, check_contrasts):
    for ax, region in zip(row, composites):
        participant_strip(ax, values[(values.contrast == contrast) & (values.region == region)],
                          f'aCAU {contrast}: {region}')
fig.tight_layout()
fig.savefig(os.path.join(out_dir, 'ppi_seed-aCAU_per-participant.png'), dpi=200)

robust_outliers(values[values.contrast.isin(check_contrasts)]).sort_values(['participant_id', 'contrast', 'region']).round(3)

## 3. Planned: aCAU coupling during correct feedback, learners vs natives

Composites, hemispheres and stages averaged per participant, then learners − natives (Welch t, d, group-label permutation p; FDR across the three composites). Then the learning-stage trend, and the group × stage ANOVA as a descriptive companion.

In [ ]:
acau = ppi['aCAU']
comp = acau[acau.network == composite_network]

# within group: is there feedback-related coupling at all?
within = gs.baseline_tests(comp[comp.contrast == 'ppifbcorrect'])
display(within[['participant_group', 'region', 'n', 'mean', 't', 'p', 'q', 'd']].round(3))

planned_level = gs.group_difference_tests(comp, contrasts=('ppifbcorrect',), n_perms=n_perms, seed=seed)
display(planned_level.round(3))

In [ ]:
trends = gs.subject_trends(acau)
comp_trends = trends[trends.network == composite_network]
planned_trend = gs.trend_difference_tests(comp_trends, contrasts=('ppifbcorrect',), n_perms=n_perms, seed=seed)
display(planned_trend.round(3))

# full battery (within-group trend vs 0, LOSO, bootstrap) per composite
battery = pd.concat([gs.trend_group_difference(comp_trends, 'ppifbcorrect', r, n_perms=n_perms, n_boot=n_boot, seed=seed)
                     for r in composites], ignore_index=True)
battery.round(3)

In [ ]:
anova = gs.group_by_stage_anova(comp, contrasts=('ppifbcorrect',))
anova.round(3)

## 4. Learners: wrong feedback, and correct vs wrong

Learners only. Does aCAU couple with auditory cortex after wrong feedback, is coupling stronger after correct than wrong feedback, and do these change with learning? Wrong trials thin out as learners improve, so final-stage `ppifbwrong` rests on fewer trials (counts in section 1). Trend tests: t-test, exact sign-flip, bootstrap CI; FDR across composites within contrast.

In [ ]:
learner_contrasts = ('ppifbwrong', 'ppifbcorrectvswrong')
learners = comp[comp.contrast.isin(learner_contrasts) & (comp.participant_group == gs.GROUPS[0])]
display(gs.baseline_tests(learners)[['contrast', 'region', 'n', 'mean', 't', 'p', 'q', 'd']].round(3))
display(gs.baseline_tests(learners, by_stage=True)[['contrast', 'region', 'learning_stage', 'n', 'mean', 't', 'p', 'q']].round(3))

# learners' wrong-feedback trials per stage (fewer as accuracy rises)
display(qc[(qc.seed == 'aCAU') & (qc.participant_group == gs.GROUPS[0])]
        .groupby(['participant_id', 'learning_stage']).n_fb_wrong.sum()
        .unstack()[list(gs.STAGE_ORDER)].describe().loc[['mean', 'min', 'max']].round(1))

# learners' trend vs 0 (trend_group_difference needs both groups, and natives mostly lack this contrast)
from scipy import stats
learner_trends = comp_trends[comp_trends.contrast.isin(learner_contrasts)
                             & (comp_trends.participant_group == gs.GROUPS[0])]
rows = []
for contrast in learner_contrasts:
    for region in composites:
        v = learner_trends.loc[(learner_trends.contrast == contrast) & (learner_trends.region == region), 'trend'].values
        t, p = stats.ttest_1samp(v, 0)
        boot = gs.bootstrap_ci(v, n_boot=n_boot, rng=np.random.default_rng(seed))
        rows.append({'contrast': contrast, 'region': region, 'n': len(v), 'mean': v.mean(), 't': t, 'p': p,
                     'permutation_p': gs.exact_sign_flip_permutation_test(v)['p_value'],
                     'ci_lower': boot['ci_lower'], 'ci_upper': boot['ci_upper']})
learner_trend = pd.DataFrame(rows).assign(q=lambda d: d.groupby('contrast').p.transform(gs.fdr))
learner_trend.round(3)

## 5. Coupling during the sound

The stimulus-locked counterpart: does aCAU couple with auditory cortex during tone processing, and does that differ by group or change with learning?

In [ ]:
display(gs.baseline_tests(comp[comp.contrast == 'ppisound'])[['participant_group', 'region', 'n', 'mean', 't', 'p', 'q']].round(3))
display(gs.group_difference_tests(comp, contrasts=('ppisound',), n_perms=n_perms, seed=seed).round(3))
gs.trend_difference_tests(comp_trends, contrasts=('ppisound',), n_perms=n_perms, seed=seed).round(3)

## 6. Seed specificity: aPUT, and aCAU − aPUT

The univariate feedback result is a caudate-vs-putamen difference, so the comparison seed is aPUT. An aCAU-specific coupling effect should show up in the per-participant aCAU − aPUT difference, not just in aCAU. The difference can be significant when neither seed is: noise the two seeds share cancels in it. The figure shows both seeds and the difference per participant, so a difference driven by one seed (or one participant) is visible.

In [ ]:
aput_comp = ppi['aPUT'][ppi['aPUT'].network == composite_network]
display(gs.group_difference_tests(aput_comp, contrasts=('ppifbcorrect', 'ppisound'), n_perms=n_perms, seed=seed).round(3))

keys = ['participant_id', 'participant_group', 'network', 'contrast', 'region', 'region_hemi', 'hemisphere',
        'learning_stage']
seed_diff = (ppi['aCAU'].merge(ppi['aPUT'], on=keys, suffixes=('_aCAU', '_aPUT'))
             .assign(beta=lambda d: d.beta_aCAU - d.beta_aPUT)[keys + ['beta']])
diff_comp = seed_diff[seed_diff.network == composite_network]
display(gs.baseline_tests(diff_comp[diff_comp.contrast.isin(['ppifbcorrect', 'ppisound'])])
        [['contrast', 'participant_group', 'region', 'n', 'mean', 't', 'p', 'q']].round(3))
display(gs.group_difference_tests(diff_comp, contrasts=('ppifbcorrect', 'ppisound'), n_perms=n_perms, seed=seed).round(3))
gs.trend_difference_tests(gs.subject_trends(diff_comp), contrasts=('ppifbcorrect', 'ppisound'),
                          n_perms=n_perms, seed=seed).round(3)

In [ ]:
# per participant (stages averaged): each seed, then the aCAU - aPUT difference
seed_values = {'aCAU': per_participant(comp), 'aPUT': per_participant(aput_comp), 'aCAU − aPUT': per_participant(diff_comp)}
panel_contrasts = ['ppisound', 'ppifbcorrect']
fig, axes = plt.subplots(len(seed_values), len(panel_contrasts) * len(composites),
                         figsize=(2.3 * len(panel_contrasts) * len(composites), 2.6 * len(seed_values)), sharey='row')
for row, (seed_label, vals) in zip(axes, seed_values.items()):
    for ax, (contrast, region) in zip(row, [(c, r) for c in panel_contrasts for r in composites]):
        participant_strip(ax, vals[(vals.contrast == contrast) & (vals.region == region)],
                          f'{seed_label}\n{contrast}: {region}')
    row[0].set_ylabel('PPI β')
fig.tight_layout()
fig.savefig(os.path.join(out_dir, 'ppi_seed-difference_per-participant.png'), dpi=200)

## 7. Exploratory: per region

The 7 auditory regions (hemispheres averaged), FDR across regions within contrast. Not part of the planned family.

In [ ]:
regions = acau[acau.network == 'auditory']
region_level = gs.group_difference_tests(regions, contrasts=('ppifbcorrect', 'ppisound'), n_perms=n_perms, seed=seed)
region_trend = gs.trend_difference_tests(trends[trends.network == 'auditory'], contrasts=('ppifbcorrect', 'ppisound'),
                                         n_perms=n_perms, seed=seed)
display(region_level.round(3))
region_trend.round(3)

## 8. Figure: aCAU coupling by stage

In [ ]:
bilat = gs.bilateral_mean(comp)
fig, axes = plt.subplots(2, len(composites), figsize=(3.2 * len(composites), 5.6), sharey='row')
for row, contrast in zip(axes, ['ppifbcorrect', 'ppisound']):
    for ax, region in zip(row, composites):
        sub = bilat[(bilat.contrast == contrast) & (bilat.region == region)]
        for offset, group in zip((-0.05, 0.05), gs.GROUPS):
            stats_ = (sub[sub.participant_group == group].groupby('learning_stage').beta
                      .agg(['mean', 'sem']).reindex(gs.STAGE_ORDER))
            ax.errorbar(np.arange(3) + offset, stats_['mean'], yerr=stats_['sem'], marker='o', capsize=3,
                        color=colors[group], label=labels[group])
        ax.axhline(0, color='0.6', lw=0.8)
        ax.set_xticks(range(3), [s.capitalize() for s in gs.STAGE_ORDER])
        ax.set_title(f'{region}: {contrast}', fontsize=9)
    row[0].set_ylabel('aCAU coupling (PPI β)')
axes[0, 0].legend(frameon=False, fontsize=8)
fig.tight_layout()
fig.savefig(os.path.join(out_dir, 'ppi_seed-aCAU_composites_by-stage.png'), dpi=200)

## 9. Save tables

In [ ]:
for name, table in {'planned_level': planned_level, 'planned_trend': planned_trend, 'planned_battery': battery,
                    'group_by_stage_anova': anova, 'seed_coupling': seed_coupling,
                    'learners_wrong-and-correct-vs-wrong_trend': learner_trend,
                    'exploratory_region_level': region_level, 'exploratory_region_trend': region_trend}.items():
    table.to_csv(os.path.join(out_dir, f'ppi_seed-aCAU_{name}.tsv'), sep='\t', index=False)
print('saved to', out_dir)